# Geometric transformations of inverted latent trajectories

One image is translated and rotated. Each version is DDIM-inverted with the same prompt, guidance, and timestep schedule. The notebook saves the latent at every inverse step and compares the transformed image's trajectory with that same geometric transform applied in latent coordinates.

This is a single-example proof of concept. `TRANSFORMS` is the list to extend later. Nothing here launches a multi-image sweep.

Run it from the repository root in the HiPerGator Jupyter session, using the `gaussian-rotation` kernel and one GPU. Matplotlib draws the figures inside the notebook and also writes them under `results/latent_trajectory/example_001/figures`. The run does not open a desktop window. That output directory is gitignored.

After the weights load, the work is one generation and three inversions.

## Stack already used in this repository

- Model: `runwayml/stable-diffusion-v1-5`
- VAE: the pipeline `AutoencoderKL`. Spatial compression is 8, so a 512 px image is a 64×64 latent. The scale factor is checked at runtime.
- Generation scheduler: `DDIMScheduler`, with `init_noise_sigma` required to be 1
- Inversion: `StableDiffusionDiffEditPipeline.invert` with `DDIMInverseScheduler`, `inpaint_strength=1`, `num_reg_steps=0`
- 50 steps, guidance 7.5, float16 weights, prompt "a photograph of an astronaut riding a horse"
- Image padding: black RGB `(0, 0, 0)`. Translations are nearest-neighbor. Positive motion is right or down. Rotations are PIL bicubic, counterclockwise, cropped with `expand=False`.


## 1. Configuration


In [ ]:
from pathlib import Path

SEED = 12345
OUTPUT_DIR = Path("results/latent_trajectory/example_001")

MODEL_ID = "runwayml/stable-diffusion-v1-5"
NUM_INFERENCE_STEPS = 50
DEVICE = "cuda"
DTYPE = "float16"
IMAGE_SIZE = 512
PROMPT = "a photograph of an astronaut riding a horse"
GUIDANCE_SCALE = 7.5
INVERSION_STRENGTH = 1.0

# 16 px is two latent pixels when the VAE scale factor is 8, so the baseline
# latent translation is an integer shift. Change this number to try another offset.
TRANSLATION_PX = 16
ROTATION_DEG = 15.0

# One entry per geometric condition. The cells below loop over this list.
# Keep it to these two until the one-image run has been checked.
TRANSFORMS = [
    {
        "name": "horizontal_translation",
        "kind": "horizontal_translation",
        "magnitude": float(TRANSLATION_PX),
        "title": f"translation, {TRANSLATION_PX} px right",
    },
    {
        "name": "rotation",
        "kind": "rotation",
        "magnitude": float(ROTATION_DEG),
        "title": f"rotation, {ROTATION_DEG:g} deg",
    },
]


## 2. Latent-space transform

`experiments/latent_geometry.py` applies the baseline map \(T_{\mathrm{latent}}\).

- A pixel shift of \(n\) becomes a latent shift of \(n / 8\). Content moves in the same direction as the image transform. Vacated sites are filled with 0. That fill is not the VAE code of a black pixel.
- An integer latent shift is a copy. A fractional shift uses nearest neighbor, matching the image translation.
- Rotation is counterclockwise about the center, one channel at a time, with a cubic spline and 0 fill. Channels are not mixed. The spline is not PIL's bicubic kernel.
- These choices are a coordinate baseline. They are not an assumption that the VAE or the inverter is equivariant.

The reported error at each inverse step is

\[
e_t=\frac{\|z'_t-T_{\mathrm{latent}}(z_t)\|_2}{\|z'_t\|_2+\epsilon}.
\]

\(z_t\) is the inversion latent of the original image. \(z'_t\) is the inversion latent of the transformed image at the same step. \(\epsilon=10^{-8}\). The normalized difference uses the same denominator with \(z_t\) in place of \(T(z_t)\).


In [ ]:
import importlib.util
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.latent_geometry import (
    EPS,
    apply_latent_transform,
    cosine_similarity,
    equivariance_error,
    fit_shared_pca,
    normalized_difference,
    project_latent_rgb,
    selected_step_indices,
)

_geom_spec = importlib.util.spec_from_file_location(
    "geometric_registration_experiment",
    ROOT / "scripts" / "run_geometric_registration_experiment.py",
)
geom = importlib.util.module_from_spec(_geom_spec)
_geom_spec.loader.exec_module(geom)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES = OUTPUT_DIR / "figures"
FIGURES.mkdir(exist_ok=True)
IMAGE_META_PATH = OUTPUT_DIR / "image_metadata.json"
META_PATH = OUTPUT_DIR / "metadata.json"
TRAJ_PATH = OUTPUT_DIR / "trajectories.pt"
DECODED_PATH = OUTPUT_DIR / "decoded.pt"
METRICS_PATH = OUTPUT_DIR / "metrics.csv"

INVERSE = None


def image_settings() -> dict:
    return {
        "model_id": MODEL_ID,
        "seed": SEED,
        "prompt": PROMPT,
        "num_inference_steps": NUM_INFERENCE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "image_size": IMAGE_SIZE,
        "dtype": DTYPE,
        "device": DEVICE,
    }


def experiment_settings() -> dict:
    return {
        **image_settings(),
        "inversion_strength": INVERSION_STRENGTH,
        "translation_px": TRANSLATION_PX,
        "rotation_deg": ROTATION_DEG,
        "transforms": [
            {"name": item["name"], "kind": item["kind"], "magnitude": item["magnitude"]}
            for item in TRANSFORMS
        ],
    }


def _read_json(path: Path) -> dict | None:
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def _write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")


def settings_match(saved: dict | None, current: dict) -> bool:
    if saved is None:
        return False
    return json.loads(json.dumps(current)) == json.loads(json.dumps(saved))


def save_show(fig: plt.Figure, name: str) -> None:
    path = FIGURES / name
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    print(path)


def load_bundle() -> dict[str, torch.Tensor]:
    if not TRAJ_PATH.is_file():
        raise FileNotFoundError(f"Missing {TRAJ_PATH}. Run the inversion cell first.")
    bundle = torch.load(TRAJ_PATH, map_location="cpu", weights_only=True)
    meta = _read_json(META_PATH)
    if not settings_match(None if meta is None else meta.get("experiment"), experiment_settings()):
        raise RuntimeError("Saved trajectories were built with different settings. Re-run the inversion cell.")
    return bundle


def require_cuda() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available. Use the gaussian-rotation kernel on a one-GPU HiPerGator notebook session."
        )


def load_or_generate_original() -> Image.Image:
    saved = _read_json(IMAGE_META_PATH)
    image_path = OUTPUT_DIR / "original.png"
    if image_path.is_file() and settings_match(None if saved is None else saved.get("image_settings"), image_settings()):
        with Image.open(image_path) as handle:
            image = handle.convert("RGB")
            image.load()
        print(f"Loaded cached image {image_path}")
        return image

    require_cuda()
    from experiments.generate_and_invert import (
        _dtype,
        _load_generation_pipeline,
        _release_pipeline,
        _sample_latent,
    )

    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    dtype = _dtype(DTYPE)
    pipe = _load_generation_pipeline(MODEL_ID, dtype, DEVICE)
    init_noise_sigma = float(pipe.scheduler.init_noise_sigma)
    if init_noise_sigma != 1.0:
        raise RuntimeError(f"DDIM init_noise_sigma is {init_noise_sigma}, not 1.")
    if pipe.vae_scale_factor != 8:
        raise RuntimeError(f"Expected VAE scale factor 8, got {pipe.vae_scale_factor}.")
    latent = _sample_latent(pipe, SEED, IMAGE_SIZE, IMAGE_SIZE, dtype)
    result = pipe(
        prompt=PROMPT,
        latents=latent.to(DEVICE),
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        height=IMAGE_SIZE,
        width=IMAGE_SIZE,
    )
    if len(result.images) != 1:
        raise RuntimeError(f"generation returned {len(result.images)} images")
    image = result.images[0].convert("RGB")
    _release_pipeline(pipe)
    image.save(image_path)
    torch.save(latent.detach().to(device="cpu", dtype=torch.float32), OUTPUT_DIR / "generation_latent.pt")
    _write_json(
        IMAGE_META_PATH,
        {
            "image_settings": image_settings(),
            "init_noise_sigma": init_noise_sigma,
            "note": "generation_latent.pt is the Gaussian passed to the sampler. It is not an inversion latent.",
        },
    )
    print(f"Generated {image_path}")
    return image


def build_images(original: Image.Image) -> dict[str, Image.Image]:
    if original.size != (IMAGE_SIZE, IMAGE_SIZE) or original.mode != "RGB":
        raise RuntimeError(f"expected RGB {IMAGE_SIZE}x{IMAGE_SIZE}, got {original.mode} {original.size}")
    images = {"original": original}
    for spec in TRANSFORMS:
        transformed = geom.apply_geometric(original, spec["kind"], spec["magnitude"])
        if transformed.size != original.size or transformed.mode != "RGB":
            raise RuntimeError(f"{spec['name']} changed the image size or mode")
        images[spec["name"]] = transformed
    original.save(OUTPUT_DIR / "original.png")
    for spec in TRANSFORMS:
        images[spec["name"]].save(OUTPUT_DIR / f"{spec['name']}.png")
    return images


def _as_int(value) -> int:
    if torch.is_tensor(value):
        return int(value.item())
    return int(value)


def epsilon_to_x0(scheduler, model_output, timestep, sample):
    prediction_type = getattr(scheduler.config, "prediction_type", "epsilon")
    if prediction_type != "epsilon":
        raise RuntimeError(f"prediction_type is {prediction_type}, not epsilon.")
    alpha = scheduler.alphas_cumprod[_as_int(timestep)].to(device=sample.device, dtype=sample.dtype)
    return (sample - (1 - alpha).sqrt() * model_output) / alpha.sqrt()


## 3. Input comparison

The original image is the seed-12345 sample from the same generation path as `experiments/generate_and_invert.py`. The translation and the rotation reuse `apply_geometric`: black padding, nearest-neighbor shifts, bicubic counterclockwise rotation cropped to 512×512.


In [ ]:
original = load_or_generate_original()
images = build_images(original)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
panels = [("original", "original")] + [(spec["name"], spec["title"]) for spec in TRANSFORMS]
for ax, (name, title) in zip(axes, panels):
    ax.imshow(np.asarray(images[name]))
    ax.set_title(title)
    ax.axis("off")
fig.suptitle("Same 512x512 RGB preprocessing. New pixels are black.")
save_show(fig, "input_comparison.png")
print(f"Expected latent translation: {TRANSLATION_PX} / 8 = {TRANSLATION_PX / 8:g} latent pixels, if the VAE scale factor is 8.")


## 4. Inversion

Each image is encoded and inverted by `StableDiffusionDiffEditPipeline.invert`. The three calls share the pipeline, prompt, guidance, strength, and step count. The RNG is reseeded before every call so the VAE posterior sample does not pick up leftover generator state from the previous image.

Diffusers stacks the inverse steps on dimension 1 and reverses that axis, so index 0 of the raw tensor is the fully noised end. The saved tensors are put back into inversion order: index 0 is the first inverse step, the one closest to the encoded image. Predicted clean latents \(\hat z_0\) are the scheduler's `pred_original_sample` at that step. The VAE round trip below uses the posterior mode and is the reconstruction check. It is a separate tensor from the stochastic encode inside `invert`.


In [ ]:
import gc

from experiments.generate_and_invert import _dtype, _load_inversion_pipeline, _release_pipeline


def _correlation(image: Image.Image, decoded: torch.Tensor) -> float:
    observed = np.asarray(image, dtype=np.float64) / 255.0
    recovered = decoded.detach().cpu().numpy().transpose(1, 2, 0)
    left = observed.ravel() - observed.mean()
    right = recovered.ravel() - recovered.mean()
    return float(left.dot(right) / (np.linalg.norm(left) * np.linalg.norm(right) + EPS))


@torch.inference_mode()
def _encode_mode(pipe, image: Image.Image) -> torch.Tensor:
    pixels = pipe.image_processor.preprocess(image, height=IMAGE_SIZE, width=IMAGE_SIZE)
    pixels = pixels.to(device=pipe.device, dtype=pipe.vae.dtype)
    latent = pipe.vae.encode(pixels).latent_dist.mode()
    return latent * pipe.vae.config.scaling_factor


@torch.inference_mode()
def _decode_latent(pipe, latent: torch.Tensor) -> torch.Tensor:
    if latent.ndim == 3:
        latent = latent.unsqueeze(0)
    latent = latent.to(device=pipe.device, dtype=pipe.vae.dtype)
    image = pipe.vae.decode(latent / pipe.vae.config.scaling_factor, return_dict=False)[0]
    return (image / 2 + 0.5).clamp(0, 1)[0].detach().to(device="cpu", dtype=torch.float32)


def _spatial(tensor: torch.Tensor) -> torch.Tensor:
    if tensor.ndim == 4 and tensor.shape[0] == 1:
        tensor = tensor[0]
    if tensor.ndim != 3:
        raise RuntimeError(f"expected one latent map, got {tuple(tensor.shape)}")
    return tensor


def _invert_one(pipe, image: Image.Image) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    records = {"post": [], "pred": [], "times": [], "used_fallback": False}
    original_step = pipe.inverse_scheduler.step

    def wrapped_step(model_output, timestep, sample, *args, **kwargs):
        out = original_step(model_output, timestep, sample, *args, **kwargs)
        pred = getattr(out, "pred_original_sample", None)
        if pred is None and isinstance(out, tuple) and len(out) > 1:
            pred = out[1]
        if pred is None:
            pred = epsilon_to_x0(pipe.inverse_scheduler, model_output, timestep, sample)
            records["used_fallback"] = True
        prev = out.prev_sample if hasattr(out, "prev_sample") else out[0]
        records["post"].append(_spatial(prev.detach().to(device="cpu", dtype=torch.float32)).clone())
        records["pred"].append(_spatial(pred.detach().to(device="cpu", dtype=torch.float32)).clone())
        records["times"].append(_as_int(timestep))
        return out

    pipe.inverse_scheduler.step = wrapped_step
    try:
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        result = pipe.invert(
            prompt=PROMPT,
            image=image,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            inpaint_strength=INVERSION_STRENGTH,
            num_reg_steps=0,
        )
    finally:
        pipe.inverse_scheduler.step = original_step

    stacked = result.latents if hasattr(result, "latents") else result
    if stacked.ndim != 5:
        raise RuntimeError(
            f"invert() returned shape {tuple(stacked.shape)}. The full trajectory, stacked on dimension 1, is required."
        )
    api = stacked[0].detach().to(device="cpu", dtype=torch.float32)
    post = torch.stack(records["post"], 0)
    pred = torch.stack(records["pred"], 0)
    times = torch.tensor(records["times"], dtype=torch.int64)
    if api.shape != post.shape or pred.shape != post.shape or times.shape[0] != post.shape[0]:
        raise RuntimeError(
            f"trajectory shapes disagree: api {tuple(api.shape)}, steps {tuple(post.shape)}, x0 {tuple(pred.shape)}"
        )
    if torch.allclose(api[0], post[0], atol=1e-2, rtol=1e-2):
        chrono = api
    elif torch.allclose(api[0], post[-1], atol=1e-2, rtol=1e-2):
        chrono = torch.flip(api, dims=(0,))
    else:
        raise RuntimeError("invert() trajectory does not match the inverse-scheduler steps.")
    if not torch.allclose(chrono, post, atol=1e-2, rtol=1e-2):
        raise RuntimeError("Reordered invert() trajectory still disagrees with the scheduler steps.")
    if records["used_fallback"]:
        print("Scheduler step did not return pred_original_sample; used the epsilon x0 formula.")
    return chrono, pred, times


def _assert_latent(name: str, tensor: torch.Tensor, steps: int) -> None:
    expected = (steps, 4, IMAGE_SIZE // 8, IMAGE_SIZE // 8)
    if tuple(tensor.shape) != expected:
        raise RuntimeError(f"{name} shape {tuple(tensor.shape)} != {expected}")
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f"{name} has non-finite values")


saved_meta = _read_json(META_PATH)
cache_ok = TRAJ_PATH.is_file() and settings_match(
    None if saved_meta is None else saved_meta.get("experiment"),
    experiment_settings(),
)

if cache_ok:
    bundle = torch.load(TRAJ_PATH, map_location="cpu", weights_only=True)
    for name in ["original"] + [spec["name"] for spec in TRANSFORMS]:
        _assert_latent(f"traj_{name}", bundle[f"traj_{name}"], NUM_INFERENCE_STEPS)
        _assert_latent(f"x0_{name}", bundle[f"x0_{name}"], NUM_INFERENCE_STEPS)
    if tuple(bundle["timesteps"].shape) != (NUM_INFERENCE_STEPS,):
        raise RuntimeError(f"timestep count {tuple(bundle['timesteps'].shape)} != {(NUM_INFERENCE_STEPS,)}")
    print(f"Loaded cached trajectories {TRAJ_PATH}")
    print(
        f"VAE scale factor {saved_meta['vae_scale_factor']}, "
        f"first timestep {saved_meta['timestep_first']}, last timestep {saved_meta['timestep_last']}"
    )
    for name, value in saved_meta["reconstruction_correlation"].items():
        print(f"VAE reconstruction correlation, {name}: {value:.4f}")
else:
    require_cuda()
    original = load_or_generate_original()
    images = build_images(original)
    dtype = _dtype(DTYPE)
    INVERSE = _load_inversion_pipeline(MODEL_ID, dtype, DEVICE)
    pipe = INVERSE
    print("VAE", type(pipe.vae).__name__, "scale_factor", pipe.vae_scale_factor, "scaling_factor", float(pipe.vae.config.scaling_factor))
    print("UNet", type(pipe.unet).__name__, "in_channels", pipe.unet.config.in_channels)
    print("scheduler", type(pipe.scheduler).__name__, "inverse_scheduler", type(pipe.inverse_scheduler).__name__)
    if pipe.vae_scale_factor != 8:
        raise RuntimeError(f"Expected VAE scale factor 8, got {pipe.vae_scale_factor}.")
    if pipe.unet.config.in_channels != 4:
        raise RuntimeError(f"Expected 4 latent channels, got {pipe.unet.config.in_channels}.")
    if type(pipe.scheduler).__name__ != "DDIMScheduler":
        raise RuntimeError(f"Expected DDIMScheduler, got {type(pipe.scheduler).__name__}.")
    if type(pipe.inverse_scheduler).__name__ != "DDIMInverseScheduler":
        raise RuntimeError(f"Expected DDIMInverseScheduler, got {type(pipe.inverse_scheduler).__name__}.")

    correlations = {}
    recon_row = []
    input_row = []
    for name, image in images.items():
        encoded = _encode_mode(pipe, image)
        if tuple(encoded.shape) != (1, 4, IMAGE_SIZE // 8, IMAGE_SIZE // 8):
            raise RuntimeError(f"{name} encode shape {tuple(encoded.shape)}")
        if not torch.isfinite(encoded).all():
            raise RuntimeError(f"{name} encode has non-finite values")
        decoded = _decode_latent(pipe, encoded)
        if tuple(decoded.shape) != (3, IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f"{name} decode shape {tuple(decoded.shape)}")
        if not torch.isfinite(decoded).all():
            raise RuntimeError(f"{name} decode has non-finite values")
        score = _correlation(image, decoded)
        correlations[name] = score
        print(f"VAE reconstruction correlation, {name}: {score:.4f}")
        if score < 0.8:
            raise RuntimeError(f"{name} VAE reconstruction correlation {score:.4f} is below 0.8")
        input_row.append(np.asarray(image))
        recon_row.append(decoded.numpy().transpose(1, 2, 0))

    fig, axes = plt.subplots(2, len(images), figsize=(4 * len(images), 8))
    for col, name in enumerate(images):
        axes[0, col].imshow(input_row[col])
        axes[0, col].set_title(f"{name} input")
        axes[1, col].imshow(recon_row[col])
        axes[1, col].set_title(f"mode decode {correlations[name]:.3f}")
        axes[0, col].axis("off")
        axes[1, col].axis("off")
    fig.suptitle("VAE posterior-mode round trip, before trusting the inversion tensors")
    save_show(fig, "vae_reconstruction.png")

    bundle = {}
    shared_times = None
    for name, image in images.items():
        print(f"Inverting {name}")
        traj, pred, times = _invert_one(pipe, image)
        _assert_latent(f"traj_{name}", traj, NUM_INFERENCE_STEPS)
        _assert_latent(f"x0_{name}", pred, NUM_INFERENCE_STEPS)
        if shared_times is None:
            shared_times = times
        elif not torch.equal(shared_times, times):
            raise RuntimeError(f"{name} timesteps differ from the original inversion")
        bundle[f"traj_{name}"] = traj
        bundle[f"x0_{name}"] = pred
        print(f"  shape {tuple(traj.shape)}  finite {bool(torch.isfinite(traj).all() and torch.isfinite(pred).all())}")
    bundle["timesteps"] = shared_times

    metadata = {
        "experiment": experiment_settings(),
        "vae_class": type(pipe.vae).__name__,
        "vae_scale_factor": int(pipe.vae_scale_factor),
        "vae_scaling_factor": float(pipe.vae.config.scaling_factor),
        "scheduler": type(pipe.scheduler).__name__,
        "inverse_scheduler": type(pipe.inverse_scheduler).__name__,
        "inversion_api": "StableDiffusionDiffEditPipeline.invert",
        "trajectory_order": "index 0 is the first inverse step",
        "timestep_first": int(shared_times[0]),
        "timestep_last": int(shared_times[-1]),
        "latent_shape": list(bundle["traj_original"].shape),
        "reconstruction_correlation": correlations,
        "latent_transform": {
            "translation": "pixel_shift / vae_scale_factor, integer copy or nearest, fill 0",
            "rotation": "per-channel scipy cubic spline, counterclockwise, fill 0, no channel mixing",
            "epsilon": EPS,
        },
    }
    torch.save(bundle, TRAJ_PATH)
    _write_json(META_PATH, metadata)
    print(f"Saved {TRAJ_PATH}")
    print(f"Timesteps {int(shared_times[0])} -> {int(shared_times[-1])} over {len(shared_times)} steps")

print("Original trajectory", tuple(bundle["traj_original"].shape), bundle["traj_original"].dtype)
print("Timestep endpoints", int(bundle["timesteps"][0]), int(bundle["timesteps"][-1]))


## 5. Latent channels

Each row pair uses one color scale per channel, shared by the original latent and the transformed image's latent. Scales are not shared across timesteps: a fully noised step would otherwise flatten the early steps. Index 0 is the first inverse step.


In [ ]:
bundle = load_bundle()
timesteps = bundle["timesteps"].tolist()
indices = selected_step_indices(len(timesteps))

for spec in TRANSFORMS:
    for index in indices:
        left = bundle["traj_original"][index].numpy()
        right = bundle[f"traj_{spec['name']}"][index].numpy()
        fig, axes = plt.subplots(2, 4, figsize=(12, 6))
        for channel in range(4):
            vmin = float(min(left[channel].min(), right[channel].min()))
            vmax = float(max(left[channel].max(), right[channel].max()))
            if vmin == vmax:
                vmax = vmin + 1e-6
            for row, (latent, label) in enumerate(((left, "original"), (right, spec["title"]))):
                im = axes[row, channel].imshow(latent[channel], vmin=vmin, vmax=vmax, cmap="coolwarm")
                axes[row, channel].set_title(f"{label}\nch {channel}")
                axes[row, channel].axis("off")
                fig.colorbar(im, ax=axes[row, channel], fraction=0.046)
        fig.suptitle(f"{spec['title']}: inversion step {index}, t={timesteps[index]}")
        save_show(fig, f"channels_{spec['name']}_step{index}.png")


## 6. Latent differences

The maps are \(z'_t - z_t\), transformed-image trajectory minus original trajectory. Within one condition, channel panels share a diverging scale across the selected timesteps, and the magnitude panels share a separate scale. Magnitude is the root sum of squares across the four channels.


In [ ]:
bundle = load_bundle()
timesteps = bundle["timesteps"].tolist()
indices = selected_step_indices(len(timesteps))

for spec in TRANSFORMS:
    diffs = []
    mags = []
    limit = 0.0
    for index in indices:
        diff = bundle[f"traj_{spec['name']}"][index].numpy() - bundle["traj_original"][index].numpy()
        mag = np.sqrt(np.sum(diff ** 2, axis=0))
        diffs.append(diff)
        mags.append(mag)
        limit = max(limit, float(np.max(np.abs(diff))))
    limit = max(limit, 1e-8)
    mag_max = max(float(mag.max()) for mag in mags)
    mag_max = max(mag_max, 1e-8)
    fig, axes = plt.subplots(len(indices), 5, figsize=(15, 3 * len(indices)))
    for row, index in enumerate(indices):
        for channel in range(4):
            im = axes[row, channel].imshow(diffs[row][channel], vmin=-limit, vmax=limit, cmap="coolwarm")
            axes[row, channel].set_title(f"t={timesteps[index]} ch {channel}")
            axes[row, channel].axis("off")
        fig.colorbar(im, ax=axes[row, :4].ravel().tolist(), fraction=0.02)
        im_mag = axes[row, 4].imshow(mags[row], vmin=0.0, vmax=mag_max, cmap="magma")
        axes[row, 4].set_title(f"t={timesteps[index]} magnitude")
        axes[row, 4].axis("off")
        fig.colorbar(im_mag, ax=axes[row, 4], fraction=0.046)
    fig.suptitle(f"{spec['title']}: z' - z. Shared scales across the selected steps.")
    save_show(fig, f"differences_{spec['name']}.png")


## 7. Decoded trajectory

The first row of each figure decodes the saved inverse latent through the VAE. Those latents are still noisy, and a high-timestep decode is only a diagnostic. The second figure decodes the scheduler's predicted clean latent \(\hat z_0\) from that same step. That panel is the model's current clean-image estimate, not a claim that the estimate matches the input.


In [ ]:
bundle = load_bundle()
meta = _read_json(META_PATH)
timesteps = bundle["timesteps"].tolist()
indices = selected_step_indices(len(timesteps))
names = ["original"] + [spec["name"] for spec in TRANSFORMS]

if DECODED_PATH.is_file() and settings_match(meta.get("experiment") if meta else None, experiment_settings()):
    decoded = torch.load(DECODED_PATH, map_location="cpu", weights_only=True)
    saved_index = decoded["indices"].tolist()
    if saved_index != indices:
        decoded = None
else:
    decoded = None

if decoded is None:
    if INVERSE is None:
        require_cuda()
        INVERSE = _load_inversion_pipeline(MODEL_ID, _dtype(DTYPE), DEVICE)
    decoded = {
        "indices": torch.tensor(indices, dtype=torch.int64),
        "timesteps": torch.tensor([timesteps[index] for index in indices], dtype=torch.int64),
    }
    for name in names:
        noisy_frames = []
        clean_frames = []
        for index in indices:
            noisy_frames.append(_decode_latent(INVERSE, bundle[f"traj_{name}"][index]))
            clean_frames.append(_decode_latent(INVERSE, bundle[f"x0_{name}"][index]))
            if not torch.isfinite(noisy_frames[-1]).all() or not torch.isfinite(clean_frames[-1]).all():
                raise RuntimeError(f"non-finite decode for {name} at step {index}")
        decoded[f"noisy_{name}"] = torch.stack(noisy_frames, 0)
        decoded[f"x0_{name}"] = torch.stack(clean_frames, 0)
        print(f"Decoded {name}: noisy {tuple(decoded[f'noisy_{name}'].shape)} x0 {tuple(decoded[f'x0_{name}'].shape)}")
    torch.save(decoded, DECODED_PATH)
    _release_pipeline(INVERSE)
    INVERSE = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _show_decodes(prefix: str, heading: str, filename: str) -> None:
    fig, axes = plt.subplots(len(names), len(indices), figsize=(4 * len(indices), 3 * len(names)))
    for row, name in enumerate(names):
        frames = decoded[f"{prefix}_{name}"]
        for col, index in enumerate(indices):
            axes[row, col].imshow(frames[col].numpy().transpose(1, 2, 0))
            axes[row, col].set_title(f"{name}\nstep {index}, t={timesteps[index]}")
            axes[row, col].axis("off")
    fig.suptitle(heading)
    save_show(fig, filename)


_show_decodes(
    "noisy",
    "Decoded inverse latents. High-noise steps are a diagnostic and can look unstructured.",
    "decoded_noisy.png",
)
_show_decodes(
    "x0",
    "Decoded predicted clean latents z0-hat from the inverse scheduler.",
    "decoded_x0.png",
)


## 8. Shared PCA

One PCA is fit on the four latent channels. Every spatial site from the original trajectory and both transformed trajectories, at every saved timestep, is one observation. That single projection, and a single RGB scale taken from those same points, is applied to every panel below. Separate projections are not fit per image or per timestep.


In [ ]:
bundle = load_bundle()
timesteps = bundle["timesteps"].tolist()
indices = selected_step_indices(len(timesteps))
names = ["original"] + [spec["name"] for spec in TRANSFORMS]
fit_on = []
for name in names:
    traj = bundle[f"traj_{name}"].numpy()
    fit_on.extend(traj[step] for step in range(traj.shape[0]))
pca = fit_shared_pca(fit_on, n_components=3)
print(f"PCA fit on {len(fit_on)} latents. RGB range of the fit: {pca['vmin']:.4f} to {pca['vmax']:.4f}")

fig, axes = plt.subplots(len(names), len(indices), figsize=(4 * len(indices), 3 * len(names)))
for row, name in enumerate(names):
    for col, index in enumerate(indices):
        rgb = project_latent_rgb(bundle[f"traj_{name}"][index].numpy(), pca)
        axes[row, col].imshow(rgb)
        axes[row, col].set_title(f"{name}\nstep {index}, t={timesteps[index]}")
        axes[row, col].axis("off")
fig.suptitle("Shared PCA RGB of the inversion latents")
save_show(fig, "pca_rgb.png")


## 9. Timestep analysis

Cosine similarity and normalized difference compare \(z_t\) with \(z'_t\) directly. Equivariance error compares \(z'_t\) with \(T_{\mathrm{latent}}(z_t)\). The dashed cosine is \(\mathrm{cosine}(z'_t, T(z_t))\), the angular version of that second comparison.

The residual figure is \(\|z'_t - T(z_t)\|\) across channels at the selected steps. A small \(e_t\) with a large direct difference would mean the latent moved in the same way as the image. This notebook does not expect that outcome. Rotation error is the weaker of the two baselines because of the spline, the 0 fill, and the fact that the VAE was not trained to be rotationally equivariant.


In [ ]:
import pandas as pd

bundle = load_bundle()
meta = _read_json(META_PATH)
scale = int(meta["vae_scale_factor"])
timesteps = bundle["timesteps"].numpy()
rows = []
for spec in TRANSFORMS:
    for step in range(bundle["traj_original"].shape[0]):
        z = bundle["traj_original"][step].numpy()
        zp = bundle[f"traj_{spec['name']}"][step].numpy()
        mapped = apply_latent_transform(z, spec["kind"], spec["magnitude"], scale)
        rows.append(
            {
                "transform": spec["name"],
                "step_index": step,
                "timestep": int(timesteps[step]),
                "cosine_z_zp": cosine_similarity(z, zp),
                "cosine_zp_tz": cosine_similarity(zp, mapped),
                "normalized_difference": normalized_difference(zp, z),
                "equivariance_error": equivariance_error(zp, mapped),
            }
        )
metrics = pd.DataFrame(rows)
metrics.to_csv(METRICS_PATH, index=False)
print(f"Wrote {METRICS_PATH}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for spec in TRANSFORMS:
    group = metrics[metrics["transform"] == spec["name"]]
    axes[0].plot(group["timestep"], group["cosine_z_zp"], label=f"{spec['title']}: cos(z, z')")
    axes[0].plot(group["timestep"], group["cosine_zp_tz"], linestyle="--", label=f"{spec['title']}: cos(z', T(z))")
    axes[1].plot(group["timestep"], group["normalized_difference"], label=spec["title"])
    axes[2].plot(group["timestep"], group["equivariance_error"], label=spec["title"])
axes[0].set_ylabel("cosine similarity")
axes[1].set_ylabel(r"$||z' - z||_2 / (||z'||_2 + \epsilon)$")
axes[2].set_ylabel(r"$||z' - T(z)||_2 / (||z'||_2 + \epsilon)$")
for ax in axes:
    ax.set_xlabel("inverse-scheduler timestep")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
direction = "increasing" if timesteps[-1] > timesteps[0] else "decreasing"
fig.suptitle(
    f"Inversion order runs from t={int(timesteps[0])} to t={int(timesteps[-1])} ({direction} timestep)."
)
save_show(fig, "timestep_metrics.png")

indices = selected_step_indices(len(timesteps))
for spec in TRANSFORMS:
    mags = []
    mag_max = 0.0
    for index in indices:
        z = bundle["traj_original"][index].numpy()
        zp = bundle[f"traj_{spec['name']}"][index].numpy()
        mapped = apply_latent_transform(z, spec["kind"], spec["magnitude"], scale)
        mag = np.sqrt(np.sum((zp - mapped) ** 2, axis=0))
        mags.append(mag)
        mag_max = max(mag_max, float(mag.max()))
    mag_max = max(mag_max, 1e-8)
    fig, axes = plt.subplots(1, len(indices), figsize=(4 * len(indices), 3.5))
    for ax, mag, index in zip(axes, mags, indices):
        im = ax.imshow(mag, vmin=0.0, vmax=mag_max, cmap="magma")
        ax.set_title(f"step {index}, t={int(timesteps[index])}")
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"{spec['title']}: spatial ||z' - T(z)||, shared scale")
    save_show(fig, f"equivariance_residual_{spec['name']}.png")


## 10. Interpretation

The measurements below are for this one image, this prompt, guidance 7.5, and 50 DDIM inverse steps. They do not establish equivariance.

What the baseline cannot absorb:

- Black pixels in the image are not latent zeros, so translation fill and rotation fill are already a source of error.
- The latent rotation is a per-channel spline. It is only a stand-in for the PIL rotation of the pixels.
- The VAE and the UNet are not trained to commute with these transforms. Classifier-free guidance ties the trajectory to the prompt as well as to the pixels.
- DDIM inversion is itself an approximation. The earlier magnitude-correlation runs in this project already show that a perfectly aligned inversion does not reproduce the original latent one-for-one.
- Decoding a noisy latent through the VAE is not a view of the image. The \(\hat z_0\) decode is the cleaner diagnostic.


In [ ]:
import pandas as pd

metrics = pd.read_csv(METRICS_PATH)
print(f"Steps: {metrics['step_index'].nunique()}. Timesteps {int(metrics['timestep'].iloc[0])} -> {int(metrics['timestep'].iloc[-1])}.")
for spec in TRANSFORMS:
    group = metrics[metrics["transform"] == spec["name"]].sort_values("step_index")
    first = group.iloc[0]
    last = group.iloc[-1]
    closer = int((group["equivariance_error"] < group["normalized_difference"]).sum())
    print(
        f"{spec['title']}: equivariance error {first['equivariance_error']:.3f} at step 0 "
        f"(t={int(first['timestep'])}) and {last['equivariance_error']:.3f} at the last step "
        f"(t={int(last['timestep'])}). Direct normalized difference {first['normalized_difference']:.3f} -> "
        f"{last['normalized_difference']:.3f}. cosine(z, z') {first['cosine_z_zp']:.3f} -> {last['cosine_z_zp']:.3f}. "
        f"cosine(z', T(z)) {first['cosine_zp_tz']:.3f} -> {last['cosine_zp_tz']:.3f}. "
        f"T(z) is closer than z on {closer} of {len(group)} steps."
    )
print(
    "Read a gap between the direct difference and the equivariance error as how much this latent-space "
    "baseline explains. A remaining error is expected, especially for the 15 degree rotation."
)
